# 09 - Model Comparison: RoBERTa Fine-Tuned vs Ollama Zero-Shot

This notebook loads all saved results from notebooks 07 and 08 and produces a comprehensive comparison.

### What we compare
| Model | Type | Training data needed |
|---|---|---|
| RoBERTa-base fine-tuned | Fine-tuned | Yes (labeled gold standard) |
| Ollama llama3.1:8b zero-shot | Zero-shot | No |

### Datasets: Manchester, Monkeypox, PHEME

### Metrics: F1 Macro, F1 Weighted, Accuracy, Precision, Recall, F1 (positive class)

## 1. Imports & Setup

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from pathlib import Path
from collections import Counter
from sklearn.metrics import confusion_matrix, precision_recall_curve, average_precision_score

# statsmodels for McNemar test — install if missing
try:
    from statsmodels.stats.contingency_tables import mcnemar
except ImportError:
    import subprocess, sys
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'statsmodels', '-q'])
    from statsmodels.stats.contingency_tables import mcnemar

plt.rcParams['figure.dpi'] = 120
plt.rcParams['font.size']  = 11

RESULTS_DIR = Path('../../results')
PREDS_DIR   = RESULTS_DIR / 'predictions'
FIGS_DIR    = RESULTS_DIR / 'figures'
FIGS_DIR.mkdir(parents=True, exist_ok=True)

DATASETS = ['manchester', 'monkeypox', 'pheme']
LABEL_CONFIGS = {
    'manchester': {'label_names': ['reliable', 'misinformation'], 'pos_label': 'misinformation', 'label_map': {'reliable': 0, 'misinformation': 1}},
    'monkeypox':  {'label_names': ['reliable', 'misinformation'], 'pos_label': 'misinformation', 'label_map': {'reliable': 0, 'misinformation': 1}},
    'pheme':      {'label_names': ['not_rumour', 'rumour'],       'pos_label': 'rumour',          'label_map': {'not_rumour': 0, 'rumour': 1}},
}

# Probability column names for PR curves (roberta prediction files)
PROB_COLS = {
    'manchester': ('prob_reliable', 'prob_misinformation'),
    'monkeypox':  ('prob_reliable', 'prob_misinformation'),
    'pheme':      ('prob_not_rumour', 'prob_rumour'),
}

COLORS = {'roberta': '#2980b9', 'zeroshot': '#e67e22'}
ALPHA_SIG = 0.05   # significance threshold

print("Setup complete.")

## 2. Load All Results

In [ ]:
def load_summaries():
    """Load summary CSVs saved by notebooks 07 and 08."""
    rows = []
    for ds in DATASETS:
        roberta_path  = PREDS_DIR / f'{ds}_roberta_summary.csv'
        zeroshot_path = PREDS_DIR / f'{ds}_zeroshot_summary.csv'

        if roberta_path.exists():
            r = pd.read_csv(roberta_path).iloc[0].to_dict()
            r['model_type'] = 'RoBERTa Fine-Tuned'
            r['dataset'] = ds
            rows.append(r)
        else:
            print(f"WARNING: {roberta_path} not found — run notebook 07 first")

        if zeroshot_path.exists():
            z = pd.read_csv(zeroshot_path).iloc[0].to_dict()
            # Normalise model name regardless of whether it says Ollama or Gemini
            z['model_type'] = 'Zero-Shot LLM'
            z['dataset'] = ds
            rows.append(z)
        else:
            print(f"WARNING: {zeroshot_path} not found — run notebook 08 first")

    return pd.DataFrame(rows)


df_summary = load_summaries()
print(f"Loaded {len(df_summary)} result rows")
print(df_summary[['dataset', 'model_type', 'test_f1_macro', 'test_accuracy']].to_string(index=False))

## 3. Main Comparison Table

In [ ]:
METRIC_COLS = ['test_f1_macro', 'test_f1_weighted', 'test_accuracy', 'test_precision', 'test_recall']

# Pivot for easy comparison
pivot = df_summary.pivot_table(
    index='dataset',
    columns='model_type',
    values=METRIC_COLS
).round(4)

print("\n=== FULL COMPARISON TABLE ===")
print(pivot.to_string())

# Delta: RoBERTa - ZeroShot (positive = RoBERTa wins)
print("\n=== DELTA (RoBERTa Fine-Tuned MINUS Zero-Shot LLM) ===")
for col in METRIC_COLS:
    try:
        delta = pivot[col]['RoBERTa Fine-Tuned'] - pivot[col]['Zero-Shot LLM']
        print(f"\n{col}:")
        print(delta.round(4).to_string())
    except KeyError:
        pass

## 3b. McNemar's Test — Statistical Significance of Prediction Differences

In [ ]:
def load_cv_std(ds):
    """Load per-fold CV results and return std of f1_macro across folds."""
    cv_path = PREDS_DIR / f'{ds}_cv_results.csv'
    if not cv_path.exists():
        return None
    df_cv = pd.read_csv(cv_path)
    return df_cv['f1_macro'].std()


metrics_to_plot = ['test_f1_macro', 'test_accuracy', 'test_precision', 'test_recall']
metric_labels   = ['F1 Macro', 'Accuracy', 'Precision\n(Macro)', 'Recall\n(Macro)']

# --- Figure: one subplot per dataset ---
fig, axes = plt.subplots(1, len(DATASETS), figsize=(7 * len(DATASETS), 6), sharey=False)

for ax, ds in zip(axes, DATASETS):
    ds_data = df_summary[df_summary['dataset'] == ds]

    roberta_row  = ds_data[ds_data['model_type'] == 'RoBERTa Fine-Tuned']
    zeroshot_row = ds_data[ds_data['model_type'] == 'Zero-Shot LLM']

    x = np.arange(len(metrics_to_plot))
    width = 0.35

    r_vals = [roberta_row[m].values[0]  if len(roberta_row)  > 0 and m in roberta_row.columns  else 0 for m in metrics_to_plot]
    z_vals = [zeroshot_row[m].values[0] if len(zeroshot_row) > 0 and m in zeroshot_row.columns else 0 for m in metrics_to_plot]

    # Error bars: use CV std for f1_macro only (RoBERTa), no CV for zero-shot
    cv_std = load_cv_std(ds)
    r_yerr = np.zeros(len(metrics_to_plot))
    z_yerr = np.zeros(len(metrics_to_plot))
    if cv_std is not None:
        r_yerr[0] = cv_std   # only f1_macro index

    bars1 = ax.bar(x - width/2, r_vals, width,
                   label='RoBERTa Fine-Tuned',
                   color=COLORS['roberta'], alpha=0.85,
                   edgecolor='black', linewidth=0.7,
                   yerr=r_yerr, capsize=6, error_kw={'elinewidth': 1.8, 'ecolor': '#1a5276'})
    bars2 = ax.bar(x + width/2, z_vals, width,
                   label='Zero-Shot LLM',
                   color=COLORS['zeroshot'], alpha=0.85,
                   edgecolor='black', linewidth=0.7,
                   yerr=z_yerr, capsize=6, error_kw={'elinewidth': 1.8, 'ecolor': '#935116'})

    # Value labels above each bar
    for bar in list(bars1) + list(bars2):
        h = bar.get_height()
        if h > 0:
            ax.text(bar.get_x() + bar.get_width() / 2, h + 0.012, f'{h:.3f}',
                    ha='center', va='bottom', fontsize=8.5, fontweight='bold', rotation=45)

    ax.set_title(ds.upper(), fontweight='bold', fontsize=15)
    ax.set_xticks(x)
    ax.set_xticklabels(metric_labels, fontsize=11)
    ax.set_ylim([0, 1.15])
    ax.set_ylabel('Score', fontsize=12)
    ax.legend(fontsize=10, loc='lower right')
    ax.grid(axis='y', alpha=0.3, linestyle='--')
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)

    # Footnote if error bars present
    if cv_std is not None:
        ax.annotate('Error bar on F1 Macro = RoBERTa 5-fold CV std',
                    xy=(0.5, -0.14), xycoords='axes fraction',
                    ha='center', fontsize=8, color='gray')

plt.suptitle('RoBERTa Fine-Tuned vs Zero-Shot LLM — Per-Dataset Comparison',
             fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()
fig_path = FIGS_DIR / 'comparison_grouped_bars.png'
plt.savefig(fig_path, dpi=150, bbox_inches='tight')
plt.show()
print(f"Saved: {fig_path}")

## 4. Grouped Bar Chart — F1 Macro Comparison

In [ ]:
# Build heatmap matrix
heatmap_cols = ['test_f1_macro', 'test_f1_weighted', 'test_accuracy', 'test_precision', 'test_recall']
heatmap_labels = ['F1 Macro', 'F1 Weighted', 'Accuracy', 'Precision', 'Recall']

rows_r, rows_z = [], []
idx_r, idx_z = [], []

for ds in DATASETS:
    r = df_summary[(df_summary['dataset'] == ds) & (df_summary['model_type'] == 'RoBERTa Fine-Tuned')]
    z = df_summary[(df_summary['dataset'] == ds) & (df_summary['model_type'] == 'Zero-Shot LLM')]
    if len(r):
        rows_r.append([r[c].values[0] if c in r.columns else np.nan for c in heatmap_cols])
        idx_r.append(f'RoBERTa | {ds}')
    if len(z):
        rows_z.append([z[c].values[0] if c in z.columns else np.nan for c in heatmap_cols])
        idx_z.append(f'Zero-Shot | {ds}')

all_rows = rows_r + rows_z
all_idx  = idx_r  + idx_z

hm_df = pd.DataFrame(all_rows, index=all_idx, columns=heatmap_labels)

fig, ax = plt.subplots(figsize=(10, max(4, len(all_idx) * 0.7)))
sns.heatmap(
    hm_df.round(3),
    annot=True, fmt='.3f', cmap='YlGnBu',
    vmin=0.5, vmax=1.0,
    linewidths=0.5, linecolor='white',
    ax=ax, annot_kws={'size': 10}
)
ax.set_title('Model Comparison Heatmap — All Metrics & Datasets', fontsize=13, fontweight='bold')
ax.set_xticklabels(ax.get_xticklabels(), rotation=20, ha='right')
plt.tight_layout()
fig_path = FIGS_DIR / 'comparison_heatmap.png'
plt.savefig(fig_path, dpi=150, bbox_inches='tight')
plt.show()
print(f"Saved: {fig_path}")

## 5. Heatmap — F1 Macro Across All Conditions

In [ ]:
fig, axes = plt.subplots(len(DATASETS), 2, figsize=(12, 5 * len(DATASETS)))

for row_idx, ds in enumerate(DATASETS):
    cfg = LABEL_CONFIGS[ds]

    roberta_pred_path  = PREDS_DIR / f'{ds}_roberta_test_predictions.csv'
    zeroshot_pred_path = PREDS_DIR / f'{ds}_zeroshot_test_predictions.csv'

    for col_idx, (pred_path, title, cmap) in enumerate([
        (roberta_pred_path,  'RoBERTa Fine-Tuned', 'Blues'),
        (zeroshot_pred_path, 'Zero-Shot LLM',      'Oranges'),
    ]):
        ax = axes[row_idx][col_idx]
        if pred_path.exists():
            df_p = pd.read_csv(pred_path)
            y_true = df_p['true_label_int'].values
            y_pred = df_p['pred_label_int'].values
            cm = confusion_matrix(y_true, y_pred)
            cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True)

            annot = np.array([
                [f"{cm[i,j]}\n({cm_norm[i,j]:.1%})" for j in range(cm.shape[1])]
                for i in range(cm.shape[0])
            ])
            sns.heatmap(
                cm_norm, annot=annot, fmt='', cmap=cmap,
                xticklabels=cfg['label_names'],
                yticklabels=cfg['label_names'],
                ax=ax, vmin=0, vmax=1
            )
            ax.set_title(f'{ds.upper()} | {title}', fontweight='bold')
            ax.set_xlabel('Predicted')
            ax.set_ylabel('True')
        else:
            ax.text(0.5, 0.5, f'No data\n{pred_path.name}',
                    ha='center', va='center', transform=ax.transAxes)
            ax.set_title(f'{ds.upper()} | {title}', fontweight='bold')

plt.suptitle('Confusion Matrices: RoBERTa Fine-Tuned vs Zero-Shot LLM', fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
fig_path = FIGS_DIR / 'comparison_confusion_matrices.png'
plt.savefig(fig_path, dpi=150, bbox_inches='tight')
plt.show()
print(f"Saved: {fig_path}")

## 6b. Precision-Recall Curves (per Dataset)

In [ ]:
print("=" * 70)
print(" WINNER ANALYSIS — F1 Macro (primary metric) + Statistical Significance")
print("=" * 70)

for ds in DATASETS:
    r = df_summary[(df_summary['dataset'] == ds) & (df_summary['model_type'] == 'RoBERTa Fine-Tuned')]
    z = df_summary[(df_summary['dataset'] == ds) & (df_summary['model_type'] == 'Zero-Shot LLM')]

    if len(r) and len(z) and 'test_f1_macro' in r.columns and 'test_f1_macro' in z.columns:
        r_f1 = r['test_f1_macro'].values[0]
        z_f1 = z['test_f1_macro'].values[0]
        delta = r_f1 - z_f1
        winner = 'RoBERTa Fine-Tuned' if delta > 0 else 'Zero-Shot LLM'
        sign = '+' if delta > 0 else ''

        # Statistical significance from McNemar (computed in section 3b)
        if ds in mcnemar_results:
            pval  = mcnemar_results[ds]['pvalue']
            is_sig = mcnemar_results[ds]['significant']
            sig_str = f"p={pval:.4f} → {'SIGNIFICANT' if is_sig else 'not significant'}"
        else:
            sig_str = "significance unknown (run section 3b first)"

        print(f"\n  {ds.upper():12s} → Winner: {winner}")
        print(f"               RoBERTa F1:    {r_f1:.4f}")
        print(f"               Zero-Shot F1:  {z_f1:.4f}")
        print(f"               Delta F1:      {sign}{delta:.4f}  ({abs(delta)*100:.2f} pp)")
        print(f"               McNemar:       {sig_str}")
    else:
        print(f"\n  {ds.upper():12s} → Missing data (run notebooks 07 and 08 first)")

print("\n" + "=" * 70)
print(" OVERALL VERDICT")
print("=" * 70)

roberta_wins = sum(
    1 for ds in DATASETS
    if (
        df_summary[(df_summary['dataset'] == ds) & (df_summary['model_type'] == 'RoBERTa Fine-Tuned')]['test_f1_macro'].values[0]
        > df_summary[(df_summary['dataset'] == ds) & (df_summary['model_type'] == 'Zero-Shot LLM')]['test_f1_macro'].values[0]
    ) if (
        len(df_summary[(df_summary['dataset'] == ds) & (df_summary['model_type'] == 'RoBERTa Fine-Tuned')]) > 0 and
        len(df_summary[(df_summary['dataset'] == ds) & (df_summary['model_type'] == 'Zero-Shot LLM')]) > 0
    )
)
print(f"\n  RoBERTa Fine-Tuned wins on {roberta_wins}/{len(DATASETS)} datasets by F1 Macro.")
sig_count = sum(1 for v in mcnemar_results.values() if v['significant'])
print(f"  Statistically significant difference on {sig_count}/{len(mcnemar_results)} tested datasets (alpha={ALPHA_SIG}).")

## 6. Confusion Matrix Side-by-Side

In [ ]:
fig, ax = plt.subplots(figsize=(14, max(4, (len(df_summary) + 1) * 0.55)))
ax.axis('off')

col_labels = ['Dataset', 'Model', 'F1 Macro', 'F1 Weighted', 'Accuracy', 'Precision', 'Recall']

table_data = []
for ds in DATASETS:
    for mtype in ['RoBERTa Fine-Tuned', 'Zero-Shot LLM']:
        row = df_summary[(df_summary['dataset'] == ds) & (df_summary['model_type'] == mtype)]
        if len(row):
            r = row.iloc[0]
            table_data.append([
                ds.capitalize(),
                mtype,
                f"{r.get('test_f1_macro', np.nan):.4f}",
                f"{r.get('test_f1_weighted', np.nan):.4f}",
                f"{r.get('test_accuracy', np.nan):.4f}",
                f"{r.get('test_precision', np.nan):.4f}",
                f"{r.get('test_recall', np.nan):.4f}",
            ])

if table_data:
    table = ax.table(
        cellText=table_data,
        colLabels=col_labels,
        cellLoc='center',
        loc='center',
        bbox=[0, 0, 1, 1]
    )
    table.auto_set_font_size(False)
    table.set_fontsize(10)

    for (row, col), cell in table.get_celld().items():
        if row == 0:
            cell.set_facecolor('#2c3e50')
            cell.set_text_props(color='white', fontweight='bold')
        elif row > 0:
            model_val = table_data[row - 1][1] if row <= len(table_data) else ''
            if 'RoBERTa' in model_val:
                cell.set_facecolor('#d6eaf8')
            else:
                cell.set_facecolor('#fdebd0')

    table.auto_set_column_width(col=list(range(len(col_labels))))
else:
    ax.text(0.5, 0.5, 'No results loaded. Run notebooks 07 and 08 first.',
            ha='center', va='center', fontsize=12, transform=ax.transAxes)

ax.set_title('Complete Results: RoBERTa Fine-Tuned vs Zero-Shot LLM', fontsize=13, fontweight='bold', pad=20)
plt.tight_layout()
fig_path = FIGS_DIR / 'comparison_final_table.png'
plt.savefig(fig_path, dpi=150, bbox_inches='tight')
plt.show()
print(f"Saved: {fig_path}")

## 7. Per-Dataset Winner Analysis

In [ ]:
print("=" * 70)
print(" WINNER ANALYSIS — F1 Macro (primary metric)")
print("=" * 70)

for ds in DATASETS:
    r = df_summary[(df_summary['dataset'] == ds) & (df_summary['model_type'] == 'RoBERTa Fine-Tuned')]
    z = df_summary[(df_summary['dataset'] == ds) & (df_summary['model_type'] == 'Ollama Zero-Shot')]

    if len(r) and len(z) and 'test_f1_macro' in r.columns and 'test_f1_macro' in z.columns:
        r_f1 = r['test_f1_macro'].values[0]
        z_f1 = z['test_f1_macro'].values[0]
        delta = r_f1 - z_f1
        winner = 'RoBERTa Fine-Tuned' if delta > 0 else 'Ollama Zero-Shot'
        sign = '+' if delta > 0 else ''
        print(f"\n  {ds.upper():12s} → Winner: {winner}")
        print(f"               RoBERTa F1:  {r_f1:.4f}")
        print(f"               Ollama F1:   {z_f1:.4f}")
        print(f"               Delta:       {sign}{delta:.4f}")
    else:
        print(f"\n  {ds.upper():12s} → Missing data (run notebooks 07 and 08 first)")

## 10. Per-Dataset Error Analysis

For each dataset we show:
- **Top 5 false negatives** — misinformation/rumour tweets that *both* models missed (true positive class, predicted negative)
- **Word frequency comparison** — most common words in misclassified vs correctly classified positive-class samples (RoBERTa)

In [ ]:
import re

def tokenize(text):
    """Simple whitespace + punctuation tokenizer; remove stopwords."""
    STOPWORDS = {
        'the','a','an','is','in','of','to','and','it','for','on','that',
        'this','with','was','are','be','at','by','from','have','has','i',
        'he','she','they','we','you','my','his','her','their','our','its',
        'as','but','or','if','not','so','up','do','did','had','about','me',
        'was','were','would','could','should','will','been','can','just',
        'than','then','when','there','here','what','who','which','no','am',
    }
    tokens = re.findall(r"[a-zA-Z']+", str(text).lower())
    return [t for t in tokens if len(t) > 2 and t not in STOPWORDS]


def top_words(texts, n=20):
    """Return the n most common words across a list of texts."""
    all_tokens = []
    for t in texts:
        all_tokens.extend(tokenize(t))
    return Counter(all_tokens).most_common(n)


def get_text_col(df):
    """Detect the tweet text column name."""
    for cand in ['cleaned_tweet', 'clean_text', 'text']:
        if cand in df.columns:
            return cand
    return df.columns[0]


for ds in DATASETS:
    cfg = LABEL_CONFIGS[ds]
    pos_int = cfg['label_map'][cfg['pos_label']]
    pos_label_name = cfg['pos_label']

    r_path = PREDS_DIR / f'{ds}_roberta_test_predictions.csv'
    z_path = PREDS_DIR / f'{ds}_zeroshot_test_predictions.csv'

    if not r_path.exists():
        print(f"\n[{ds.upper()}] RoBERTa prediction file not found — skipping.")
        continue

    df_r = pd.read_csv(r_path)
    text_col = get_text_col(df_r)

    # --- False negatives in RoBERTa (positive class predicted as negative) ---
    fn_mask = (df_r['true_label_int'] == pos_int) & (df_r['pred_label_int'] != pos_int)
    df_fn_r = df_r[fn_mask].copy()

    # Also check zero-shot FNs for overlap
    if z_path.exists():
        df_z = pd.read_csv(z_path)
        z_text_col = get_text_col(df_z)
        n_align = min(len(df_r), len(df_z))
        df_r_al = df_r.iloc[:n_align].reset_index(drop=True)
        df_z_al = df_z.iloc[:n_align].reset_index(drop=True)

        both_fn_mask = (
            (df_r_al['true_label_int'] == pos_int) &
            (df_r_al['pred_label_int'] != pos_int) &
            (df_z_al['pred_label_int'] != pos_int)
        )
        df_both_fn = df_r_al[both_fn_mask].copy()
    else:
        df_both_fn = df_fn_r

    print("\n" + "=" * 70)
    print(f" ERROR ANALYSIS — {ds.upper()}  (positive class: {pos_label_name})")
    print("=" * 70)
    print(f"  RoBERTa false negatives (FN): {len(df_fn_r)}")
    print(f"  Both models missed:           {len(df_both_fn)}")

    # Top 5 tweets missed by both models
    top5 = df_both_fn.head(5)
    if len(top5) == 0:
        top5 = df_fn_r.head(5)
        print(f"\n  (No shared FNs; showing RoBERTa-only FNs)")

    print(f"\n  Top false negatives ('{pos_label_name}' predicted as negative):")
    for i, (_, row) in enumerate(top5.iterrows(), 1):
        tweet = str(row[text_col])[:120].replace('\n', ' ')
        print(f"    {i}. [{tweet}...]")

    # Word frequency: misclassified vs correctly classified positive samples
    tp_mask  = (df_r['true_label_int'] == pos_int) & (df_r['pred_label_int'] == pos_int)
    fn_texts = df_fn_r[text_col].tolist()
    tp_texts = df_r[tp_mask][text_col].tolist()

    fn_words = top_words(fn_texts, n=10)
    tp_words = top_words(tp_texts, n=10)

    print(f"\n  Top words in MISCLASSIFIED '{pos_label_name}' (FN, n={len(fn_texts)}):")
    for w, c in fn_words:
        print(f"    {w:<20s}  {c}")

    print(f"\n  Top words in CORRECTLY CLASSIFIED '{pos_label_name}' (TP, n={len(tp_texts)}):")
    for w, c in tp_words:
        print(f"    {w:<20s}  {c}")

In [ ]:
# Visualise word frequency comparison (FN vs TP) per dataset
TOP_N = 12

fig, axes = plt.subplots(len(DATASETS), 2,
                          figsize=(16, 4.5 * len(DATASETS)),
                          constrained_layout=True)

for row_idx, ds in enumerate(DATASETS):
    cfg = LABEL_CONFIGS[ds]
    pos_int = cfg['label_map'][cfg['pos_label']]
    pos_label_name = cfg['pos_label']

    r_path = PREDS_DIR / f'{ds}_roberta_test_predictions.csv'
    if not r_path.exists():
        for c in range(2):
            axes[row_idx][c].set_visible(False)
        continue

    df_r = pd.read_csv(r_path)
    text_col = get_text_col(df_r)

    fn_mask = (df_r['true_label_int'] == pos_int) & (df_r['pred_label_int'] != pos_int)
    tp_mask = (df_r['true_label_int'] == pos_int) & (df_r['pred_label_int'] == pos_int)

    fn_texts = df_r[fn_mask][text_col].tolist()
    tp_texts = df_r[tp_mask][text_col].tolist()

    for col_idx, (texts, title, color, count) in enumerate([
        (fn_texts, f'False Negatives (RoBERTa)\nmissed {pos_label_name}', '#e74c3c', len(fn_texts)),
        (tp_texts, f'True Positives (RoBERTa)\ncorrect {pos_label_name}',  '#27ae60', len(tp_texts)),
    ]):
        ax = axes[row_idx][col_idx]
        words = top_words(texts, n=TOP_N)
        if not words:
            ax.text(0.5, 0.5, 'No data', ha='center', va='center', transform=ax.transAxes)
            ax.set_title(f'{ds.upper()} | {title}', fontweight='bold', fontsize=10)
            continue

        wds, counts = zip(*words)
        y_pos = np.arange(len(wds))
        ax.barh(y_pos, counts, color=color, alpha=0.75, edgecolor='black', linewidth=0.5)
        ax.set_yticks(y_pos)
        ax.set_yticklabels(wds, fontsize=10)
        ax.invert_yaxis()
        ax.set_xlabel('Frequency', fontsize=10)
        ax.set_title(f'{ds.upper()} | {title} (n={count})', fontweight='bold', fontsize=10)
        ax.grid(axis='x', alpha=0.3, linestyle='--')
        ax.spines['top'].set_visible(False)
        ax.spines['right'].set_visible(False)

plt.suptitle('Word Frequency: False Negatives vs True Positives (RoBERTa, positive class)',
             fontsize=13, fontweight='bold')
fig_path = FIGS_DIR / 'comparison_error_word_freq.png'
plt.savefig(fig_path, dpi=150, bbox_inches='tight')
plt.show()
print(f"Saved: {fig_path}")

## 8. Overall Summary Table

In [ ]:
fig, ax = plt.subplots(figsize=(14, max(4, (len(df_summary) + 1) * 0.55)))
ax.axis('off')

col_labels = ['Dataset', 'Model', 'F1 Macro', 'F1 Weighted', 'Accuracy', 'Precision', 'Recall']

table_data = []
for ds in DATASETS:
    for mtype in ['RoBERTa Fine-Tuned', 'Ollama Zero-Shot']:
        row = df_summary[(df_summary['dataset'] == ds) & (df_summary['model_type'] == mtype)]
        if len(row):
            r = row.iloc[0]
            table_data.append([
                ds.capitalize(),
                mtype,
                f"{r.get('test_f1_macro', np.nan):.4f}",
                f"{r.get('test_f1_weighted', np.nan):.4f}",
                f"{r.get('test_accuracy', np.nan):.4f}",
                f"{r.get('test_precision', np.nan):.4f}",
                f"{r.get('test_recall', np.nan):.4f}",
            ])

if table_data:
    table = ax.table(
        cellText=table_data,
        colLabels=col_labels,
        cellLoc='center',
        loc='center',
        bbox=[0, 0, 1, 1]
    )
    table.auto_set_font_size(False)
    table.set_fontsize(10)

    for (row, col), cell in table.get_celld().items():
        if row == 0:
            cell.set_facecolor('#2c3e50')
            cell.set_text_props(color='white', fontweight='bold')
        elif row > 0:
            model_val = table_data[row - 1][1] if row <= len(table_data) else ''
            if 'RoBERTa' in model_val:
                cell.set_facecolor('#d6eaf8')
            else:
                cell.set_facecolor('#fdebd0')

    table.auto_set_column_width(col=list(range(len(col_labels))))
else:
    ax.text(0.5, 0.5, 'No results loaded. Run notebooks 07 and 08 first.',
            ha='center', va='center', fontsize=12, transform=ax.transAxes)

ax.set_title('Complete Results: RoBERTa Fine-Tuned vs Ollama Zero-Shot (llama3.1)', fontsize=13, fontweight='bold', pad=20)
plt.tight_layout()
fig_path = FIGS_DIR / 'comparison_final_table.png'
plt.savefig(fig_path, dpi=150, bbox_inches='tight')
plt.show()
print(f"Saved: {fig_path}")

## 9. Save Master Results CSV

In [ ]:
master_path = RESULTS_DIR / 'master_results.csv'
df_summary.to_csv(master_path, index=False)
print(f"Master results saved: {master_path}")
print("\nAll comparisons complete!")
print("\nFigures saved:")
for f in sorted(FIGS_DIR.glob('comparison_*.png')):
    print(f"  {f.name}")